In [110]:
"""
Hourly Data Assimilation & Spatial Interpolation using Universal Kriging (Elevation Drift)

Overview:
----------
This script builds hourly, gridded predictor fields (temperature, RH, PLP, and MRoS proxy)
on a 1-km DEM grid using Universal Kriging with elevation as an external drift term.
The workflow auto-fits a per-hour variogram and interpolates each variable from available
station, satellite (IMERG), and citizen-science (MRoS) observations.

Pipeline:
----------
1. CONFIG:
   - Defines variables, kriging model settings, and hourly time window.
   - Uses a spherical variogram with automatic parameter fitting.

2. UTILITIES:
   - Handles time indexing, CRS management, and DEM grid setup.

3. DATA:
   - Loads hourly station, IMERG, and MRoS parquet data.
   - Filters all observations to the DEM area of interest (AOI).

4. DEM Utilities:
   - Ensures each observation has an elevation from the DEM.

5. INTERPOLATION:
   - Performs per-hour Universal Kriging with elevation as an external drift.
   - Automatically fits variograms (PyKrige internal auto-fit).
   - Chunked prediction across the DEM grid to manage memory.
   - Local neighborhood (kriging_neighbors) limits the number of nearby stations used.

6. HOURLY LOOP:
   - Iterates over each hour and variable to build a full xarray Dataset.

7. OUTPUTS:
   - Saves CF-compliant NetCDF file with gridded hourly predictors.
   - Generates optional static quicklook PNG maps with station overlays.

Key Parameters:
----------------
- min_points:     Minimum observations per variable required for kriging.
- kriging_neighbors:  Number of nearest observations used per grid-cell prediction.
- variogram_model: Type of spatial autocorrelation model ("spherical" default).
- variogram_strategy: "auto" lets PyKrige fit parameters for each hour dynamically.

Outputs:
--------
- CF-compliant NetCDF:  hourly_predictors_1km_kriging_*.nc
- Quicklook PNG maps:   /outputs/hourly_pipeline/maps/

Notes:
-------
This version replaces earlier lapse-rate detrending with a physically informed
Universal Kriging approach that treats elevation as a continuous external drift,
reducing over-smoothing while preserving local terrain-driven gradients.
"""


# ============================ IMPORTS ============================
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
import rasterio as rio
from rasterio.warp import transform_bounds, calculate_default_transform, reproject, Resampling
from rasterio.transform import xy as rio_xy, rowcol as rio_rowcol
import xarray as xr
from pyproj import CRS, Transformer
import matplotlib.pyplot as plt
from tqdm import tqdm


# Kriging
from pykrige.uk import UniversalKriging


In [111]:
BASE_DIR = Path().resolve().parent
print("BASE_DIR:", BASE_DIR)

CONFIG = {
    # Time windows
    "wy_start":  "2024-10-01T00:00:00Z",
    "wy_end":    "2025-05-31T23:59:59Z",
    "test_start": "2025-03-30T00:00:00Z",   # narrow test window first
    "test_end":   "2025-04-02T23:00:00Z",
    # "test_start": "2024-10-01T00:00:00Z",   # Entire window
    # "test_end":   "2025-05-31T23:59:59Z",

    # Paths
    "dem_path":  BASE_DIR / "DEM_1km_clipped_v2_rpj.tif",   # ensure projected (meters)
    "out_dir":   BASE_DIR / "outputs/hourly_pipeline",

    # Projection fallback if DEM CRS is geographic
    "proj_fallback": "EPSG:26911",  # UTM 11N

    # Data inputs (hourly parquets produced upstream)
    "stations_parquet": BASE_DIR / "outputs/hourly_pipeline/hourly_data/stations_hourly.parquet",
    "imerg_parquet":    BASE_DIR / "outputs/hourly_pipeline/hourly_data/imerg_hourly.parquet",
    "mros_parquet":     BASE_DIR / "outputs/hourly_pipeline/hourly_data/mros_hourly.parquet",

    # Variables
    "variables": [
        ("temp_air",       "station"),
        ("temp_dew",       "station"),
        ("temp_wet",       "station"),
        ("rh",             "station"),
        ("mros_plp_proxy", "mros"),
        ("plp",            "imerg"),
    ],

    "min_points": {  # per-variable minimum points
        "temp_air": 4, "temp_dew": 4, "temp_wet": 4, "rh": 4,
        "mros_plp_proxy": 2, "plp": 1
    },

    # Kriging/variogram
    "variogram_model": "spherical",        # keep spherical as default
    "kriging_chunk_size": 2000,             # predict grid in chunks; how many grid points get kriged per iteration
    "kriging_neighbors": 20,  # number of nearest points used in each local prediction (controls smoothing and speed)

    "variogram_strategy": "auto", 
}

OUT_DIR = Path(CONFIG["out_dir"]); OUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_DIR: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype


In [112]:
# ============================ UTILITIES ============================
def to_utc(dt_series: pd.Series) -> pd.DatetimeIndex:
    """Force timestamps to UTC, making naive → UTC-naive assumed in UTC."""
    dt = pd.to_datetime(dt_series, errors="coerce", utc=True)
    # If dt_series had naive datetimes and pandas assumed local, .tz_convert('UTC') not needed.
    return dt

def hourly_index(start_iso: str, end_iso: str) -> pd.DatetimeIndex:
    return pd.date_range(start=pd.to_datetime(start_iso), end=pd.to_datetime(end_iso),
                         freq="H", tz="UTC")

def print_time(ts):
    return pd.to_datetime(ts).strftime("%Y-%m-%d %H:%MZ")

In [113]:
# --------------------- Load DEM ------------------------

# --------------------- Load DEM ------------------------

def grid_centers(profile):
    T = profile["transform"]
    xs = T.c + (np.arange(profile["width"]) + 0.5) * T.a
    ys = T.f + (np.arange(profile["height"]) + 0.5) * T.e
    X, Y = np.meshgrid(xs, ys)
    return np.column_stack([X.ravel(), Y.ravel()])

with rio.open(CONFIG["dem_path"]) as src:
    dem_crs = src.crs
    if not dem_crs or not dem_crs.is_projected:
        print(f"DEM is geographic ({dem_crs}); reprojecting to {CONFIG['proj_fallback']} ...")
        dst_crs = CONFIG["proj_fallback"]
        transform, width, height = calculate_default_transform(src.crs, dst_crs, src.width, src.height, *src.bounds)
        kwargs = src.meta.copy(); kwargs.update({"crs": dst_crs, "transform": transform, "width": width, "height": height})
        dem_data = np.empty((height, width), dtype=np.float32)
        reproject(
            source=rio.band(src, 1), destination=dem_data,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=transform, dst_crs=dst_crs,
            resampling=Resampling.bilinear,
        )
        dem_profile = kwargs
        proj_crs = CRS.from_user_input(dst_crs)
    else:
        dem_profile = src.profile
        dem_data = src.read(1)
        proj_crs = dem_crs

# Mask NaN values in DEM data
nan_mask = np.isnan(dem_data)

# Mask the NaN values in the grid elevation
dem_data[nan_mask] = np.nan

# Generate grid coordinates
grid_xy = grid_centers(dem_profile)

# Convert DEM data into 1D array and mask NaNs
grid_elev = dem_data.ravel()
grid_elev[nan_mask.ravel()] = np.nan  # Mask NaN regions in the elevation array

# Ensure grid dimensions match
H, W = dem_profile["height"], dem_profile["width"]
T = dem_profile["transform"]
cols = np.arange(W); rows = np.arange(H)
x_centers = np.array([rio_xy(T, 0, c, offset="center")[0] for c in cols])
y_centers = np.array([rio_xy(T, r, 0, offset="center")[1] for r in rows])

print(f"DEM CRS: {proj_crs}, pixel ~{abs(T.a):.2f} m | grid {W} x {H}")
print(f"DEM CRS: {proj_crs}, pixel size: {abs(dem_profile['transform'].a):.2f} m")

# Mask the invalid grid points and keep only valid ones
valid_points = ~np.isnan(grid_elev)
grid_xy_valid = grid_xy[valid_points]
grid_elev_valid = grid_elev[valid_points]

# Optionally, print to ensure we're filtering out NaN correctly
print(f"Valid grid points count: {len(grid_xy_valid)}")


def load_dem_and_aoi(dem_path: str):
    with rio.open(dem_path) as src:
        dem_crs = CRS.from_wkt(src.crs.to_wkt()) if src.crs else None
        bounds = src.bounds
        aoi_wgs84 = transform_bounds(src.crs, "EPSG:4326",
                                     bounds.left, bounds.bottom, bounds.right, bounds.top,
                                     densify_pts=21)
    aoi_poly = box(aoi_wgs84[0], aoi_wgs84[1], aoi_wgs84[2], aoi_wgs84[3])
    return dem_path, dem_crs, aoi_poly

dem_path, dem_crs, aoi_poly = load_dem_and_aoi(CONFIG["dem_path"])

DEM CRS: EPSG:26911, pixel ~961.82 m | grid 146 x 260
DEM CRS: EPSG:26911, pixel size: 961.82 m
Valid grid points count: 35278


In [114]:
# ============================ DATA LOADING ============================

# Load hourly parquets (already generated upstream)
st_hr   = pd.read_parquet(CONFIG["stations_parquet"])
imerg_hr = pd.read_parquet(CONFIG["imerg_parquet"])
mros_hr  = pd.read_parquet(CONFIG["mros_parquet"])

# Time to UTC and filter window
for df, time_col in [(st_hr, "hour_utc"), (imerg_hr, "hour_utc"), (mros_hr, "hour_utc")]:
    df[time_col] = pd.to_datetime(df[time_col], utc=True, errors="coerce").dt.floor("h")

HOURS = hourly_index(CONFIG["test_start"], CONFIG["test_end"])  # inclusive hourly range

# Filter to AOI bbox in lon/lat

def filter_points_to_aoi(df: pd.DataFrame, aoi_poly) -> pd.DataFrame:
    g = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs="EPSG:4326")
    poly = gpd.GeoSeries([aoi_poly], crs="EPSG:4326").iloc[0]
    mask = g.intersects(poly)
    return df.loc[mask.values].drop(columns=["geometry"], errors="ignore")

st_hr   = filter_points_to_aoi(st_hr, aoi_poly)
imerg_hr= filter_points_to_aoi(imerg_hr, aoi_poly)
mros_hr    = filter_points_to_aoi(mros_hr, aoi_poly)

print(len(st_hr), len(imerg_hr), len(mros_hr))


C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\2193629452.py:9: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  return pd.date_range(start=pd.to_datetime(start_iso), end=pd.to_datetime(end_iso),


327193 2017560 7442


In [115]:
# ============================= 4) DEM utils =============================

def add_dem_elev_if_missing(st_df: pd.DataFrame, profile, proj_crs) -> pd.DataFrame:
    """Fill missing station elevations by nearest-neighbor sampling of DEM."""
    if "elev" not in st_df.columns:
        st_df = st_df.copy(); st_df["elev"] = np.nan
    need = st_df["elev"].isna()
    if not need.any():
        return st_df

    tf = Transformer.from_crs("EPSG:4326", proj_crs, always_xy=True)
    xx, yy = tf.transform(st_df.loc[need, "lon"].values, st_df.loc[need, "lat"].values)
    rr, cc = rio_rowcol(profile["transform"], xx, yy, op=round)
    rr = np.clip(rr, 0, profile["height"] - 1)
    cc = np.clip(cc, 0, profile["width"]  - 1)
    st_df = st_df.copy(); st_df.loc[need, "elev"] = dem_data[rr, cc]
    return st_df

In [116]:
# ============================= 5) INTERPOLATION =============================

def _project_lonlat_to_xy(lon, lat, dst_crs):
    tf = Transformer.from_crs("EPSG:4326", dst_crs, always_xy=True)
    return tf.transform(lon, lat)


def _select_variogram_params(strategy: str):
    """Return variogram parameters depending on strategy."""
    if strategy == "fixed":
        return CONFIG.get("variogram_fixed_params", None)
    # 'auto' mode: let PyKrige fit automatically
    return None

def krige_with_elev_drift(hour_points: pd.DataFrame, grid_xy: np.ndarray, grid_elev: np.ndarray,
                          proj_crs, value_col: str, station_elev_col: str,
                          min_points: int) -> np.ndarray:
    """Universal Kriging with elevation as an external drift (PyKrige-compatible version)."""
    
    # Filter the points with non-null values for the variable we want to interpolate
    pts = hour_points.dropna(subset=[value_col, "lon", "lat"]).copy()
    
    if pts.empty or pts[value_col].notna().sum() < min_points:
        print(f"Not enough points for kriging ({value_col})")
        return np.full(grid_elev.shape, np.nan, dtype=np.float32)

    # Ensure the elevation column is present, if not, set to zero
    if station_elev_col not in pts.columns:
        pts[station_elev_col] = 0.0

    # Project station coordinates (lon, lat) to the DEM's CRS
    px, py = _project_lonlat_to_xy(pts["lon"].values, pts["lat"].values, proj_crs)
    vals = pts[value_col].values.astype(float)
    stn_z = pts[station_elev_col].values.astype(float)

    # Print the shapes of the coordinates, grid, and external drift values
    print("Observation points (px, py):", px.shape, py.shape)
    print("Prediction grid (grid_xy):", grid_xy.shape)
    print("Station elevations (stn_z):", stn_z.shape)

    # Check if dimensions of station elevations match the coordinates
    if len(px) != len(stn_z):
        print(f"Error: Shape mismatch between station coordinates and elevations. "
              f"Expected {len(px)} and {len(stn_z)}")
        return np.full(grid_elev.shape, np.nan, dtype=np.float32)

    # Check if the prediction grid elevation dimensions match
    if grid_xy.shape[0] != grid_elev.shape[0]:
        print(f"Error: Mismatch between prediction grid size ({grid_xy.shape[0]}) "
              f"and external drift size ({grid_elev.shape[0]})")
        return np.full(grid_elev.shape, np.nan, dtype=np.float32)

    # Sample elevation values from grid_elev
    print("Sample grid elevation values:", grid_elev[:10])  # Show first 10 values from grid_elev
    
    # Select variogram parameters based on strategy
    vparams = _select_variogram_params(CONFIG["variogram_strategy"])

    try:
        # Universal Kriging with elevation as external drift term
        UK = UniversalKriging(
            px, py, vals,
            variogram_model=CONFIG["variogram_model"],
            variogram_parameters=vparams,
            drift_terms=["external_Z"],  # Elevation as external drift term
            external_drift=stn_z,  # Elevation at the observation points
            external_drift_x=grid_xy[:, 0],  # X coordinates of prediction grid
            external_drift_y=grid_xy[:, 1]   # Y coordinates of prediction grid
        )
        print("Observation points (px, py):", px.shape, py.shape)
        print("Prediction grid (grid_xy):", grid_xy.shape)
        print("Station elevations (stn_z):", stn_z.shape)

    except Exception as e:
        print(f"UniversalKriging init failed: {e}")
        return np.full(grid_elev.shape, np.nan, dtype=np.float32)

    z_pred = np.full(len(grid_xy), np.nan, dtype=np.float32)
    n_closest = CONFIG.get("kriging_neighbors", 24)
    chunk_size = int(CONFIG["kriging_chunk_size"])

    # Prediction in chunks to avoid memory overload
    for s in range(0, len(grid_xy), chunk_size):
        e = min(s + chunk_size, len(grid_xy))
        try:
            # Perform kriging predictions with the external drift for the prediction grid
            chunk_z, _ = UK.execute("points", grid_xy[s:e, 0], grid_xy[s:e, 1],
                                    external_drift=grid_elev[s:e],  # Use grid elevations for prediction points
                                    n_closest_points=n_closest)
            z_pred[s:e] = np.asarray(chunk_z, dtype=np.float32)
        except Exception as err:
            print(f"Chunk {s}-{e} failed: {err}")
    return z_pred




In [117]:
# ============================= 6) HOURLY LOOP =============================

coords = {"time": HOURS, "y": y_centers, "x": x_centers}
var_names = [v[0] for v in CONFIG["variables"]]
data_vars = {name: np.full((len(HOURS), H, W), np.nan, dtype=np.float32) for name in var_names}

for ti, t in enumerate(tqdm(HOURS, desc="Hourly surfaces", ncols=88)):
    st_t   = st_hr[st_hr["hour_utc"] == t]
    imerg_t= imerg_hr[imerg_hr["hour_utc"] == t]
    mros_t = mros_hr[mros_hr["hour_utc"] == t]

    # Ensure station elevs present for lapse
    st_t = add_dem_elev_if_missing(st_t, dem_profile, proj_crs)

    for name, src in CONFIG["variables"]:
        min_pts = CONFIG["min_points"].get(name, 3)

        if src == "station":
            if name not in st_t.columns:
                continue
            pts = st_t[["lon", "lat", "elev", name]].dropna(subset=[name])
        elif src == "imerg":
            pts = imerg_t.rename(columns={"plp": name})[["lon", "lat", name]].assign(elev=0.0)
        elif src == "mros":
            pts = mros_t.rename(columns={"mros_plp_proxy": name})[["lon", "lat", name]].assign(elev=0.0)
        else:
            continue

        if pts[name].notna().sum() < min_pts:
            print(f"    {name}: insufficient points ({pts[name].notna().sum()} < {min_pts})")
            continue
        
        # HANDLING FOR MRoS PROXY (discrete / constant cases): fractionalize and introduce small random noise to preserve ordinal meaning but allow nonzero semivariance
        if name == "mros_plp_proxy":
            pts[name] = pts[name]/100.0 + np.random.uniform(-0.02, 0.02, len(pts))
            print(f"MRoS proxy variance @ {t}: {pts[name].var():.4f}")

        try:

            vals = krige_with_elev_drift(
                hour_points=pts,
                grid_xy=grid_xy_valid,
                grid_elev=grid_elev_valid,
                proj_crs=proj_crs,
                value_col=name,
                station_elev_col="elev",
                min_points=min_pts,
            )

            if len(pts):
                print(f"    min={pts[name].min():.3f}, max={pts[name].max():.3f}, "
                    f"mean={pts[name].mean():.3f}, var={pts[name].var():.6f}")
            if name == "mros_plp_proxy":
                vals = np.clip(vals * 100.0, 0.0, 100.0)

        except Exception as e:
            print(f"Kriging failed for {name} @ {t}: {e}")
            vals = np.full(grid_elev.shape, np.nan)
        
        data_vars[name][ti, :, :] = vals.reshape(H, W)

Hourly surfaces:   0%|                                           | 0/96 [00:00<?, ?it/s]

Observation points (px, py): (40,) (40,)
Prediction grid (grid_xy): (35278, 2)
Station elevations (stn_z): (40,)
Sample grid elevation values: [1235.8988 1218.146  1260.2549 1301.958  1371.8804 1425.0746 1452.4297
 1483.9264 1619.4459 1815.167 ]
UniversalKriging init failed: External drift dimensions do not match provided x- and y-coordinate dimensions.
    min=-1.278, max=17.389, mean=4.993, var=22.284314


ValueError: cannot reshape array of size 35278 into shape (260,146)

In [ ]:
# ============================= 7) SAVE =============================

ds = xr.Dataset(
    {**{k: xr.DataArray(v, coords=coords, dims=("time","y","x")) for k, v in data_vars.items()},
     "elev": xr.DataArray(dem_data.astype(np.float32), coords={"y": y_centers, "x": x_centers}, dims=("y","x"))},
    attrs={
        "title": "Hourly predictor stacks on 1-km grid (Universal Kriging w/ elevation drift)",
        "interpolation_method": "Universal Kriging (external Z drift = DEM elevation)",
        "variogram_model": CONFIG["variogram_model"],
        "variogram_strategy": CONFIG["variogram_strategy"],
        "test_window": f"{CONFIG['test_start']} → {CONFIG['test_end']}",
    }
)

# CF mapping and CRS
ds = ds.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)
ds = ds.rio.write_crs(dem_profile["crs"], grid_mapping_name="spatial_ref")
ds = ds.rio.write_transform(dem_profile["transform"])
for v in ds.data_vars:
    ds[v].attrs.setdefault("grid_mapping", "spatial_ref")
A = dem_profile["transform"]
ds.attrs["GeoTransform"] = f"{A.c} {A.a} {A.b} {A.f} {A.d} {A.e}"

# Ensure time is tz-naive before NetCDF
if hasattr(ds.indexes.get("time", None), "tz") and ds.indexes["time"].tz is not None:
    ds = ds.assign_coords(time=ds.indexes["time"].tz_localize(None))

# Write compressed NetCDF
out_nc = OUT_DIR / "hourly_predictors_1km_kriging_v3_test.nc"

def _chunks_for(da):
    if da.dims == ("time", "y", "x"):
        return (min(24, da.sizes["time"]), min(256, da.sizes["y"]), min(256, da.sizes["x"]))
    if da.dims == ("y", "x"):
        return (min(256, da.sizes["y"]), min(256, da.sizes["x"]))
    return None

encoding = {}
for name, da in ds.data_vars.items():
    ch = _chunks_for(da)
    encoding[name] = ({"zlib": True, "complevel": 4, "chunksizes": ch}
                      if ch is not None else {"zlib": True, "complevel": 4} if da.ndim > 0 else {})

ds.to_netcdf(out_nc, engine="netcdf4", encoding=encoding)
print(f"Wrote {out_nc} (compressed).")
ds.close()


Wrote C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\hourly_predictors_1km_kriging_v3_test.nc (compressed).


In [ ]:
# -------------------- Quick Plotting ------------------------------------

from pyproj import CRS

def quicklook_hour(
    ds, t, st_t, mros_t, out_png,
    vars_to_show=("plp","mros_plp_proxy","temp_air","temp_dew","temp_wet","rh")
):
    # match time index
    times_ds = pd.to_datetime(ds.time.values).floor("h")
    t_floor  = pd.to_datetime(t).floor("h")
    if t_floor not in times_ds.values:
        print(f"No matching time {t_floor} in dataset for quicklook.")
        return
    ti = int(np.where(times_ds == t_floor)[0][0])

    # axes extent (xmin, xmax, ymin, ymax)
    xvals = ds["x"].values
    yvals = ds["y"].values
    xmin, xmax = float(np.min(xvals)), float(np.max(xvals))
    ymin, ymax = float(np.min(yvals)), float(np.max(yvals))
    extent = [xmin, xmax, ymin, ymax]

    keep = [v for v in vars_to_show if v in ds.data_vars]
    if not keep:
        print("No matching variables to plot.")
        return
    ncols, nrows = 3, int(np.ceil(len(keep)/3))

    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5*ncols, 3.8*nrows), squeeze=False)
    fig.suptitle(f"Quicklook @ {t_floor:%Y-%m-%d %H:%MZ}", fontsize=14)

    # dataset CRS (fallback to configured)
    target_crs = ds.rio.crs or CRS.from_user_input(CONFIG["proj_fallback"])
    tf = Transformer.from_crs("EPSG:4326", target_crs, always_xy=True)

    # --- project & CLIP stations ---
    st_x = np.empty(0)
    st_y = np.empty(0)
    if len(st_t):
        sx, sy = tf.transform(st_t["lon"].values, st_t["lat"].values)
        sx = np.asarray(sx); sy = np.asarray(sy)
        smask = (sx >= xmin) & (sx <= xmax) & (sy >= ymin) & (sy <= ymax) & np.isfinite(sx) & np.isfinite(sy)
        st_x, st_y = sx[smask], sy[smask]

    # --- project & CLIP MRoS ---
    mo_x = np.empty(0)
    mo_y = np.empty(0)
    if len(mros_t):
        mx, my = tf.transform(mros_t["lon"].values, mros_t["lat"].values)
        mx = np.asarray(mx); my = np.asarray(my)
        mmask = (mx >= xmin) & (mx <= xmax) & (my >= ymin) & (my <= ymax) & np.isfinite(mx) & np.isfinite(my)
        mo_x, mo_y = mx[mmask], my[mmask]

    for i, var in enumerate(keep):
        ax = axes[i // ncols, i % ncols]
        arr = ds[var].isel(time=ti).values

        # color scaling
        if var in ("plp", "mros_plp_proxy", "rh"):
            im = ax.imshow(arr, origin="upper", extent=extent, aspect="equal", vmin=0, vmax=100)
        else:
            im = ax.imshow(arr, origin="upper", extent=extent, aspect="equal")

        ax.set_title(var)
        ax.set_xlabel("Easting (km)")
        ax.set_ylabel("Northing (km)")
        ax.ticklabel_format(style="plain")   # disable 1e6 scientific format
        ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
        ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])

        # overlay
        if st_x.size:
            ax.scatter(st_x, st_y, s=15, c="white", edgecolor="k",
                    marker="o", linewidths=0.5, label="Stations", zorder=3)
        if mo_x.size:
            ax.scatter(mo_x, mo_y, s=25, c="red", edgecolor="k",
                    marker="^", linewidths=0.6, label="MRoS", zorder=3)

        ax.legend(loc="upper right", frameon=True, fontsize=8)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)

    # turn off any leftover panels
    for j in range(len(keep), nrows*ncols):
        axes[j // ncols, j % ncols].axis("off")

    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    fig.savefig(out_png, dpi=200)
    plt.close(fig)
    print(
        f"Saved quicklook: {out_png} | plotted {st_x.size} stations, {mo_x.size} MRoS (clipped to DEM)"
    )


# -------------------- Loop --------------------

quick_dir = Path(CONFIG["out_dir"]) / "maps"
quick_dir.mkdir(parents=True, exist_ok=True)

# day = "2025-03-04"
# all_times = pd.to_datetime(ds.time.values).floor("h")  # dataset times
# mask = all_times.normalize() == pd.to_datetime(day)
# sample_hours = all_times[mask]
sample_hours = pd.to_datetime(ds.time.values)[::max(1, len(ds.time)//20)]
# print(f"Found {len(sample_hours)} timesteps on {day}")

for t in sample_hours:
    t_floor = pd.to_datetime(t).floor("h")  # tz-naive

    # Ensure obs times are made tz-naive before comparison
    st_t   = st_hr[st_hr["hour_utc"].dt.tz_convert(None).dt.floor("h") == t_floor]
    mros_t = mros_hr[mros_hr["hour_utc"].dt.tz_convert(None).dt.floor("h") == t_floor]

    print(f"[{t_floor}] Stations: {len(st_t)}, MRoS: {len(mros_t)}")

    quicklook_hour(ds, t_floor, st_t, mros_t,
                   out_png=quick_dir / f"test3_kriging_quick_{print_time(t_floor).replace(':','-')}.png")


[2025-03-30 00:00:00] Stations: 40, MRoS: 0


C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.

Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\test3_kriging_quick_2025-03-30 00-00Z.png | plotted 40 stations, 0 MRoS (clipped to DEM)
[2025-03-30 04:00:00] Stations: 39, MRoS: 0


C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.

Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\test3_kriging_quick_2025-03-30 04-00Z.png | plotted 39 stations, 0 MRoS (clipped to DEM)
[2025-03-30 08:00:00] Stations: 40, MRoS: 0


...

Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\test3_kriging_quick_2025-04-02 12-00Z.png | plotted 40 stations, 1 MRoS (clipped to DEM)
[2025-04-02 16:00:00] Stations: 40, MRoS: 0


C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.

Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\test3_kriging_quick_2025-04-02 16-00Z.png | plotted 40 stations, 0 MRoS (clipped to DEM)
[2025-04-02 20:00:00] Stations: 40, MRoS: 5


C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_yticklabels([f"{y/1000:.0f}" for y in ax.get_yticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:69: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([f"{x/1000:.0f}" for x in ax.get_xticks()])
C:\Users\EmmaGolub\AppData\Local\Temp\ipykernel_37152\3764455212.py:70: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.

Saved quicklook: C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs\hourly_pipeline\maps\test3_kriging_quick_2025-04-02 20-00Z.png | plotted 40 stations, 5 MRoS (clipped to DEM)
